# CSA08 Assignment 4 – Task 4: File and Exception Forensics

**Student Name:** Darshan A  
**Register Number:** 1973260011  
**Course Code:** CSA0812  
**Marks:** 7 | **Bloom’s Level:** K4 – Analyse

## Aim
To investigate defects in a library book-return logging program, demonstrate the defects using three daily files, repair the program, and state rules that prevent the same problems.

## Task 4 overview
The original program loses earlier days because it opens the log in write mode. It also writes book names without newline separators and may not close the log if a write operation raises an exception.


## Task 4.1 – Locate the Defects (2 Marks)

| Defect | Why it happens | Result |
|---|---|---|
| Earlier days are lost | `open("returns.log", "w")` truncates the existing file whenever the function runs. | Each new day replaces the previous day's returns. Only the latest day's entries remain. |
| Book names are joined together | `log.write(b)` writes exactly the supplied text and does not add a newline automatically. | Consecutive book names appear on the same line and are difficult to distinguish. |
| File may remain open after a write failure | `log.close()` is reached only if the loop completes normally. If `write()` raises an exception, execution jumps out before `close()`. | The program does not explicitly close the file on that error path. The operating system may keep the file handle open until it is cleaned up. |
| Error is not saved | `process()` catches only `FileNotFoundError` and only prints `missing`. Other errors are not logged. | There is no permanent record explaining the failure. |
| Input file is not managed with `with` | `open(day_file).read()` opens the file without a context manager. | The program does not explicitly guarantee prompt closure of the input file. |

**Main idea:** `w` replaces a file, `a` adds to a file, and `write()` does not add line breaks unless `\n` is included.


## Task 4.2 – Construct Evidence (2 Marks)

The next cell creates three sample daily files and runs the original program exactly in the way described in the question. It then displays the log after each day so that the overwrite defect and missing line breaks can be seen.


In [1]:
from pathlib import Path

# Create three sample daily return files
sample_folder = Path("library_returns")
sample_folder.mkdir(exist_ok=True)

daily_returns = {
    "day1.txt": "Python Basics\nData Science",
    "day2.txt": "Computer Networks\nDatabase Systems",
    "day3.txt": "Artificial Intelligence\nWeb Programming"
}

for filename, content in daily_returns.items():
    with open(sample_folder / filename, "w", encoding="utf-8") as file:
        file.write(content)

print("Three daily return files created.")
print(sorted(path.name for path in sample_folder.glob("*.txt")))


# Original program from the assignment – intentionally contains defects
def save_returns_original(books):
    log = open("returns_original.log", "w")
    for book in books:
        log.write(book)
    log.close()


def process_original(day_file):
    try:
        books = open(day_file, encoding="utf-8").read().split("\n")
        save_returns_original(books)
    except FileNotFoundError:
        print("missing")


# Run the original program for three days and display the log each time
for filename in ["day1.txt", "day2.txt", "day3.txt"]:
    process_original(sample_folder / filename)

    print("\nLog after", filename)
    with open("returns_original.log", "r", encoding="utf-8") as file:
        print(repr(file.read()))


Three daily return files created.
['day1.txt', 'day2.txt', 'day3.txt']

Log after day1.txt
'Python BasicsData Science'

Log after day2.txt
'Computer NetworksDatabase Systems'

Log after day3.txt
'Artificial IntelligenceWeb Programming'


### Observation from the three-day test

- After Day 1, the log contains only Day 1's books.
- After Day 2, Day 1's entries disappear because write mode replaces the file.
- After Day 3, only Day 3's entries remain.
- `repr()` makes the output easy to inspect and shows that there are no newline characters between the book names.

This test demonstrates both the overwrite defect and the missing-separator defect.


### Demonstrate what happens when `write()` fails

In the original function, `log.close()` appears after the loop. The following controlled example attempts to write a non-string value. `write()` raises `TypeError`, so the next line (`log.close()`) is skipped. The test then checks whether the file object is still open and closes it safely for the notebook.


In [2]:
# Controlled demonstration of a write failure in the original design

log = open("write_failure_demo.log", "w", encoding="utf-8")

try:
    log.write("First book")
    log.write(None)  # Deliberate error: write() requires a string
    log.close()      # This line is skipped when TypeError occurs

except TypeError as error:
    print("Write failed:", type(error).__name__)
    print("Is the file still open?", not log.closed)

finally:
    # Close the file here so the demonstration itself leaves no open file.
    if not log.closed:
        log.close()

print("Is the file closed after cleanup?", log.closed)


Write failed: TypeError
Is the file still open? True
Is the file closed after cleanup? True


## Task 4.3 – Repair the Program (2 Marks)

The corrected version makes these changes:

1. Uses append mode (`a`) so earlier days are preserved.
2. Adds `\n` after each returned book so every book is stored on a separate line.
3. Uses `with open(...)` for both input and output files so they are closed automatically.
4. Uses an error log to save the filename and reason when a file is missing or another file operation fails.
5. Handles each daily file separately so one missing file does not stop processing the remaining days.


In [3]:
from pathlib import Path


def save_returns(books, log_filename="returns_fixed.log"):
    # Append mode keeps returns already saved on earlier days.
    with open(log_filename, "a", encoding="utf-8") as log:
        for book in books:
            book = book.strip()
            if book:
                log.write(book + "\n")


def process(day_file, log_filename="returns_fixed.log",
            error_filename="returns_errors.log"):
    try:
        # Read the daily file safely.
        with open(day_file, "r", encoding="utf-8") as file:
            books = file.read().splitlines()

        # Add this day's returns to the end of the main log.
        save_returns(books, log_filename)
        print("Processed:", day_file)

    except FileNotFoundError as error:
        message = f"File not found: {day_file} | {error}"
        with open(error_filename, "a", encoding="utf-8") as error_log:
            error_log.write(message + "\n")
        print("Saved missing-file error:", day_file)

    except (PermissionError, UnicodeDecodeError, OSError) as error:
        message = f"Could not process: {day_file} | {type(error).__name__}: {error}"
        with open(error_filename, "a", encoding="utf-8") as error_log:
            error_log.write(message + "\n")
        print("Saved file-processing error:", type(error).__name__)


# Start with fresh output files for a repeatable demonstration
for output_file in ["returns_fixed.log", "returns_errors.log"]:
    Path(output_file).write_text("", encoding="utf-8")

# Process all three daily files
for filename in ["day1.txt", "day2.txt", "day3.txt"]:
    process(sample_folder / filename)

print("\nCORRECTED RETURNS LOG")
print("---------------------")
with open("returns_fixed.log", "r", encoding="utf-8") as file:
    print(file.read())


Processed: library_returns/day1.txt
Processed: library_returns/day2.txt
Processed: library_returns/day3.txt

CORRECTED RETURNS LOG
---------------------
Python Basics
Data Science
Computer Networks
Database Systems
Artificial Intelligence
Web Programming



### Test the error log with a missing daily file

A fourth call uses a filename that does not exist. The processor should save the problem in `returns_errors.log` and continue without terminating the notebook.


In [4]:
# A missing daily file is recorded instead of stopping the program
process(sample_folder / "day4_missing.txt")

print("\nERROR LOG")
print("---------")
with open("returns_errors.log", "r", encoding="utf-8") as file:
    print(file.read())


Saved missing-file error: library_returns/day4_missing.txt

ERROR LOG
---------
File not found: library_returns/day4_missing.txt | [Errno 2] No such file or directory: 'library_returns/day4_missing.txt'



## Task 4.4 – Prevent Recurrence (1 Mark)

**Rule 1 – Choose the correct file mode.** Use `w` only when a new file should replace the old contents. Use `a` when new daily records must be added without deleting earlier records.

**Rule 2 – Always manage files with a context manager.** Use `with open(...)` so files close automatically, including when an exception occurs. Also write a newline (`\n`) after each record when separate lines are required.

These two rules prevent accidental loss of earlier records, joined-up entries and files being left open after an error.


## Alternative Concept and Complexity Analysis

An alternative to `with open(...)` is to open and close files manually inside `try/finally`.

### Alternative pseudocode

```python
file = open("returns.log", "a", encoding="utf-8")
try:
    for book in books:
        file.write(book + "\\n")
finally:
    file.close()
```

The `finally` block runs even if writing raises an exception, so the file is closed. However, `with open(...)` is shorter and automatically performs the same resource-management task.

| Factor | `with open(...)` | Manual `try/finally` |
|---|---|---|
| Time complexity | O(n) for n books | O(n) for n books |
| Extra space | O(1) while writing one book at a time | O(1) while writing one book at a time |
| File closure | Automatic when the block exits | Must be written in `finally` |
| If a write fails | The context manager closes the file | The finally block closes the file if correctly written |
| Risk of coding mistake | Lower | Higher because close logic must be written correctly |

**Open-file comparison:** In the corrected `with` version, the file is closed when the block exits, even after an error. In a correctly written manual `try/finally` version, it is also closed after the `finally` block runs. In the original flawed version, the output file may remain open when `write()` fails because `close()` is skipped. The input file is also not explicitly managed with a context manager.

Both corrected approaches have the same O(n) time complexity and O(1) extra working space for line-by-line writing. The difference is safety and code simplicity, not asymptotic performance.


## Automatic Verification Checks

The final cell checks that all six books from the three days appear in the repaired log, each on a separate line, that the original flawed log contains only the final day's returns, and that the missing file was recorded in the error log.


In [5]:
# Verify the repaired output and the demonstrated defects

with open("returns_fixed.log", "r", encoding="utf-8") as file:
    fixed_lines = file.read().splitlines()

with open("returns_original.log", "r", encoding="utf-8") as file:
    original_text = file.read()

with open("returns_errors.log", "r", encoding="utf-8") as file:
    error_text = file.read()

assert len(fixed_lines) == 6
assert "Python Basics" in fixed_lines
assert "Web Programming" in fixed_lines
assert "Artificial Intelligence" in fixed_lines
assert "Python Basics" not in original_text
assert "Artificial Intelligence" in original_text
assert "day4_missing.txt" in error_text

print("All verification checks passed.")
print("Books in corrected log:", len(fixed_lines))
print("Each book is stored on a separate line.")
print("Original log demonstrates the overwrite defect.")
print("Missing file was saved in the error log.")


All verification checks passed.
Books in corrected log: 6
Each book is stored on a separate line.
Original log demonstrates the overwrite defect.
Missing file was saved in the error log.


## Task 4 – Conclusion

The original library program had three main file-handling defects: write mode removed earlier days' data, missing newline characters joined book names together, and a write failure could skip the explicit close operation. The repaired program uses append mode, writes each book on a separate line, manages files with `with` statements and records file errors in a separate log. The three-day test and verification checks demonstrate that all returned books are preserved and a missing daily file does not stop the remaining processing.
